# 🧹 Clean Bot على Google Colab

تشغيل **سيرفر FastAPI** (تنظيف المانجا بـ LaMa + كشف النصوص) و**بوت الديسكورد** داخل Colab.

**قبل ما تبدأ:**
1. Runtime ← Change runtime type ← **T4 GPU** (أو أي GPU).
2. جهّز القيم دي (تقدر تحطها في **Secrets** 🔑 في الشريط اليمين بنفس الأسماء، أو هتتسأل عليها في الخطوة 6):
   - `DISCORD_TOKEN` و `CLIENT_ID` (من Discord Developer Portal)
   - `ADMIN_USER_ID` (الـ ID بتاعك على ديسكورد)
   - `SUPABASE_URL` و `SUPABASE_KEY`
   - `GOOGLE_CLIENT_ID` و `GOOGLE_CLIENT_SECRET` و `GOOGLE_REFRESH_TOKEN` و `GOOGLE_OUTPUT_FOLDER_ID`
   - `DISCORD_BOT_API_KEY` (اختياري — لو فاضي هيتولد تلقائي)
3. شغّل الخلايا **بالترتيب** من فوق لتحت.

> ⚠️ Colab بيقفل الجلسة بعد فترة خمول، ووقتها البوت والسيرفر بيقفوا. ومع كل تشغيل جديد الرابط العام بيتغير، والخلية بتعيد كتابة إعدادات البوت تلقائياً.


## 1) التحقق من الـ GPU والـ Python

In [ ]:
!nvidia-smi
import sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2) الإعدادات

In [ ]:
REPO_URL = "https://github.com/Nader-Geek/clean-bot.git"
BRANCH = "main"              # غيّرها لـ "arena/bfefd804-clean-bot" لو عايز نسخة الفرع
REPO_DIR = "/content/clean-bot"

# عدد الموديلات اللي بتشتغل بالتوازي. T4 (15GB) → 1 أو 2 أحسن علشان ما يحصلش OOM.
CONCURRENT_MODELS = 2

API_PORT = 10100


## 3) تحميل المشروع

In [ ]:
import os, re, pathlib
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

# ضبط عدد الموديلات المتوازية (التعديل داخل نسخة Colab بس، مش بيتسجل في Git)
p = pathlib.Path("app/profiles/master.conf")
p.write_text(re.sub(r"(?m)^concurrent_models\s*=.*$", f"concurrent_models = {CONCURRENT_MODELS}", p.read_text(encoding="utf-8")), encoding="utf-8")
!grep -n "concurrent_models" app/profiles/master.conf


## 4) تثبيت مكتبات Python
Colab فيه PyTorch + CUDA جاهزين، فالخلية دي بتثبت الباقي بس. لو ظهر تعارض في numpy/opencv اعمل **Runtime → Restart session** وشغّل الخلية تاني.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q python-dotenv


## 5) Node.js وتثبيت مكتبات البوت

In [ ]:
import subprocess, shutil
def node_major():
    if not shutil.which("node"):
        return 0
    out = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    return int(out.lstrip("v").split(".")[0])

if node_major() < 18:
    print("Installing Node.js 20 ...")
    !curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
    !sudo apt-get install -y -qq nodejs > /dev/null 2>&1
print("Node major:", node_major())
!cd discord-bot && npm install --silent


## 6) إدخال الأسرار وكتابة ملفات `.env`

In [ ]:
import getpass, secrets as pysecrets
from google.colab import userdata

REQUIRED = ["DISCORD_TOKEN", "CLIENT_ID", "ADMIN_USER_ID", "SUPABASE_URL", "SUPABASE_KEY",
            "GOOGLE_CLIENT_ID", "GOOGLE_CLIENT_SECRET", "GOOGLE_REFRESH_TOKEN", "GOOGLE_OUTPUT_FOLDER_ID"]
OPTIONAL = ["DISCORD_BOT_API_KEY"]

vals = {}
for k in REQUIRED + OPTIONAL:
    try:
        v = userdata.get(k)
    except Exception:
        v = None
    if not v:
        hint = " (اختياري - Enter للتخطي)" if k in OPTIONAL else ""
        v = getpass.getpass(f"{k}{hint}: ").strip()
    vals[k] = v

missing = [k for k in REQUIRED if not vals[k]]
if missing:
    raise SystemExit(f"⛔ قيم ناقصة: {missing}")

API_KEY = vals["DISCORD_BOT_API_KEY"] or pysecrets.token_urlsafe(32)

# .env الخاص بالسيرفر (Python)
server_env = (
    f"SUPABASE_URL={vals['SUPABASE_URL']}\n"
    f"SUPABASE_KEY={vals['SUPABASE_KEY']}\n"
    f"GOOGLE_CLIENT_ID={vals['GOOGLE_CLIENT_ID']}\n"
    f"GOOGLE_CLIENT_SECRET={vals['GOOGLE_CLIENT_SECRET']}\n"
    f"GOOGLE_REFRESH_TOKEN={vals['GOOGLE_REFRESH_TOKEN']}\n"
    f"GOOGLE_OUTPUT_FOLDER_ID={vals['GOOGLE_OUTPUT_FOLDER_ID']}\n"
    f"API_HOST=0.0.0.0\n"
    f"API_PORT={API_PORT}\n"
    f"DEBUG=false\n"
    f"DISCORD_BOT_API_KEY={API_KEY}\n"
    f"API_KEY={API_KEY}\n"
)
pathlib.Path(".env").write_text(server_env, encoding="utf-8")
os.makedirs("logs", exist_ok=True)
print("✅ .env written (values hidden)")


## 7) تشغيل السيرفر (FastAPI) على المنفذ 10100

In [ ]:
import time, urllib.request
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
server_log = open("logs/server.log", "a")
api_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", str(API_PORT)],
    stdout=server_log, stderr=subprocess.STDOUT, env=env, cwd=REPO_DIR,
)

def wait_for(url, timeout=180):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if api_proc.poll() is not None:
            raise RuntimeError("السيرفر وقف — شوف logs/server.log")
        try:
            with urllib.request.urlopen(url, timeout=3) as r:
                return r.read().decode()
        except Exception:
            time.sleep(2)
    raise TimeoutError(url)

print(wait_for(f"http://127.0.0.1:{API_PORT}/api/health"))


## 8) فتح نفق عام (Cloudflare Tunnel) علشان البوت يوصل للسيرفر

In [ ]:
if not os.path.exists("cloudflared"):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

tunnel_log = open("logs/tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{API_PORT}", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT, cwd=REPO_DIR,
)

PUBLIC_URL = None
t0 = time.time()
while time.time() - t0 < 90 and not PUBLIC_URL:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("logs/tunnel.log").read())
    if m:
        PUBLIC_URL = m.group(0)
    else:
        time.sleep(2)
print("🌐 Public API URL:", PUBLIC_URL)


## 9) كتابة `.env` للبوت وتشغيله

In [ ]:
bot_env = (
    f"DISCORD_TOKEN={vals['DISCORD_TOKEN']}\n"
    f"CLIENT_ID={vals['CLIENT_ID']}\n"
    f"ADMIN_USER_ID={vals['ADMIN_USER_ID']}\n"
    f"SUPABASE_URL={vals['SUPABASE_URL']}\n"
    f"SUPABASE_KEY={vals['SUPABASE_KEY']}\n"
    f"PANEL_CLEANER_API_URL={PUBLIC_URL}\n"
    f"PANEL_CLEANER_API_KEY={API_KEY}\n"
)
pathlib.Path("discord-bot/.env").write_text(bot_env, encoding="utf-8")

bot_log = open("logs/bot.log", "a")
bot_proc = subprocess.Popen(["node", "index.js"], stdout=bot_log, stderr=subprocess.STDOUT,
                            cwd=os.path.join(REPO_DIR, "discord-bot"))
time.sleep(15)
print("Bot running" if bot_proc.poll() is None else "⛔ Bot stopped — شوف logs/bot.log")
!tail -n 20 logs/bot.log


## 10) اختبار السيرفر من الرابط العام

In [ ]:
for ep in ["/api/health", "/api/ready", "/api/device"]:
    try:
        with urllib.request.urlopen(PUBLIC_URL + ep, timeout=20) as r:
            print(ep, r.status, r.read().decode()[:300])
    except urllib.error.HTTPError as e:
        print(ep, e.code, e.read().decode()[:300])


## 11) متابعة اللوج

In [ ]:
!tail -n 60 logs/server.log

## 12) إيقاف كل الخدمات

In [ ]:
for proc in [bot_proc, tunnel_proc, api_proc]:
    if proc.poll() is None:
        proc.terminate()
print("Stopped.")
